# Lending Club EDA -- F14 -- EDA Governance, Intelligence, Synthesis & Reporting

**Status: built.** See the cell map below for what's actually in this notebook.

## What this notebook covers

The closing synthesis: every decision made across notebooks 01-13 in one place (what was dropped and why, what was kept and why, what's still open), a feature-readiness table for Phase 1, and a log of exactly what data_cleaning/ does to get from data/02_interim to data/03_processed.

## Where this fits

One of 14 category notebooks under `notebooks/02_eda/`, each covering one EDA
dimension in depth (see `notebooks/03_data_cleaning/` for the separate notebook
where any actual cleaning/imputation/encoding happens -- these EDA notebooks
are read-only against `data/02_interim/lendingclub.duckdb` and never modify
or clean the data themselves). Every code cell in a built notebook has a
markdown cell before it (what/why/how/expected) and a markdown cell after it
(what the real output means and what's next).


## Cell map

The capstone notebook -- pulls findings from all 13 EDA notebooks (01-13)
into one governance document: what's known, what's uncertain, and what the
cleaning-notebook rebuild (task 11) and Phase 1 should do about it. Every
number below is recomputed fresh in this notebook, not copy-pasted from
earlier notebooks, so it stays accurate even if an earlier notebook is
revised later.

| # | What it does |
|---|---|
| 1 | Connect; dataset headline numbers, one place |
| 2 | Feature inventory -- every retained feature, IV, and disposition recommendation |
| 3 | What's statistically confirmed vs. still just descriptive |
| 4 | Known caveats and limitations, gathered in one place |
| 5 | Action items for the `03_data_cleaning` rebuild |
| 6 | Action items and open questions for Phase 1 |

## Cell 1 -- headline numbers

**What / why:** anyone opening this notebook first should see the dataset's
shape and target definition before anything else -- this is the "if you read
nothing else" cell.

**How:** recompute the core counts directly from the interim DuckDB file.

**Expect:** the same headline numbers established in notebook 01/ingestion:
~2.26M raw rows, ~1.35M matured, ~1.2M in the 2013-2017 window, ~20.5% bad
rate.

In [1]:
import os, duckdb, pandas as pd, numpy as np
ASSETS_TABLES = "../../data/04_assets/tables"
ASSETS_PLOTS = "../../data/04_assets/plots"
os.makedirs(ASSETS_TABLES, exist_ok=True)
os.makedirs(ASSETS_PLOTS, exist_ok=True)
con = duckdb.connect("../../data/02_interim/lendingclub.duckdb", read_only=True)

n_raw = con.sql("SELECT count(*) FROM raw_mat").fetchone()[0]
n_matured = con.sql("SELECT count(*) FROM matured").fetchone()[0]
n_windowed, bad_rate = con.sql("SELECT count(*), avg(is_bad) FROM windowed").fetchone()

headline = pd.DataFrame([
    {"metric": "raw rows", "value": f"{n_raw:,}"},
    {"metric": "matured (finished-outcome) rows", "value": f"{n_matured:,}"},
    {"metric": "modeling window", "value": "2013-2017"},
    {"metric": "windowed rows", "value": f"{n_windowed:,}"},
    {"metric": "bad rate (windowed)", "value": f"{bad_rate:.1%}"},
    {"metric": "target definition", "value": "is_bad=1: Charged Off / Default; is_bad=0: Fully Paid"},
])
print(headline.to_string(index=False))
headline.to_csv(os.path.join(ASSETS_TABLES, "eda14_headline.csv"), index=False)


                         metric                                                 value
                       raw rows                                             2,260,701
matured (finished-outcome) rows                                             1,348,099
                modeling window                                             2013-2017
                  windowed rows                                             1,195,879
            bad rate (windowed)                                                 20.5%
              target definition is_bad=1: Charged Off / Default; is_bad=0: Fully Paid


**What the output shows:**
```
metric                                                 value
                       raw rows                                             2,260,701
matured (finished-outcome) rows                                             1,348,099
                modeling window                                             2013-2017
                  windowed rows                                             1,195,879
            bad rate (windowed)                                                 20.5%
              target definition is_bad=1: Charged Off / Default; is_bad=0: Fully Paid
```
Confirms the dataset's shape is stable across every notebook that's queried
it -- 1,195,879 rows, 20.5% bad rate, consistent from
notebook 01 through this one.

**Next:** the feature inventory -- what's actually being carried forward into
`data/03_processed/`, and how strong each feature is.

## Cell 2 -- feature inventory

**What / why:** consolidating every retained numeric and categorical feature
with its IV (the standard credit-scoring strength metric) into one table is
the single most useful artifact this notebook produces -- it's the reference
anyone doing Phase 1 feature selection should start from, instead of
re-deriving IV from scratch.

**How:** recompute IV for every retained feature using the same WOE-binning
approach as notebook 04 (10 bins for numeric via NTILE, raw categories for
categorical).

**Expect:** `grade` and `int_rate` at the top (already established as the
strongest predictors, per notebooks 04, 08, and 13's definitional-not-causal
finding), most other features in the "medium" or "weak" conventional IV
bands (0.02-0.3).

In [2]:
def iv_numeric(col, bins=10):
    q = con.sql(f"""
        WITH binned AS (
            SELECT NTILE({bins}) OVER (ORDER BY TRY_CAST({col} AS DOUBLE)) AS bin, is_bad
            FROM windowed WHERE TRY_CAST({col} AS DOUBLE) IS NOT NULL
        )
        SELECT bin, count(*) n, sum(is_bad) n_bad, count(*)-sum(is_bad) n_good FROM binned GROUP BY 1
    """).df()
    total_bad, total_good = q["n_bad"].sum(), q["n_good"].sum()
    good_pct = np.clip(q["n_good"]/total_good, 1e-6, None)
    bad_pct = np.clip(q["n_bad"]/total_bad, 1e-6, None)
    woe = np.log(good_pct/bad_pct)
    return float(((good_pct-bad_pct)*woe).sum())

def iv_categorical(col):
    q = con.sql(f"SELECT {col} AS cat, count(*) n, sum(is_bad) n_bad, count(*)-sum(is_bad) n_good FROM windowed WHERE {col} IS NOT NULL GROUP BY 1").df()
    total_bad, total_good = q["n_bad"].sum(), q["n_good"].sum()
    good_pct = np.clip(q["n_good"]/total_good, 1e-6, None)
    bad_pct = np.clip(q["n_bad"]/total_bad, 1e-6, None)
    woe = np.log(good_pct/bad_pct)
    return float(((good_pct-bad_pct)*woe).sum())

NUMERIC_FEATURES = ["int_rate", "dti", "annual_inc", "fico_range_low", "revol_util",
                     "revol_bal", "total_acc", "open_acc", "mort_acc", "delinq_2yrs",
                     "inq_last_6mths", "pub_rec", "tot_cur_bal", "avg_cur_bal",
                     "bc_open_to_buy", "acc_open_past_24mths", "mo_sin_old_rev_tl_op", "num_actv_rev_tl"]
CATEGORICAL_FEATURES = ["grade", "term", "home_ownership", "verification_status", "purpose"]

inv_rows = [{"feature": c, "type": "numeric", "IV": round(iv_numeric(c), 4)} for c in NUMERIC_FEATURES]
inv_rows += [{"feature": c, "type": "categorical", "IV": round(iv_categorical(c), 4)} for c in CATEGORICAL_FEATURES]
inventory = pd.DataFrame(inv_rows).sort_values("IV", ascending=False)

def iv_band(v):
    if v < 0.02: return "not useful"
    if v < 0.1: return "weak"
    if v < 0.3: return "medium"
    if v < 0.5: return "strong"
    return "suspicious (check for leakage)"
inventory["band"] = inventory["IV"].apply(iv_band)
print(inventory.to_string(index=False))
inventory.to_csv(os.path.join(ASSETS_TABLES, "eda14_inventory.csv"), index=False)


             feature        type     IV       band
               grade categorical 0.4726     strong
            int_rate     numeric 0.4627     strong
                term categorical 0.1809     medium
      fico_range_low     numeric 0.1156     medium
                 dti     numeric 0.0747       weak
acc_open_past_24mths     numeric 0.0668       weak
      bc_open_to_buy     numeric 0.0556       weak
 verification_status categorical 0.0548       weak
         avg_cur_bal     numeric 0.0508       weak
            mort_acc     numeric 0.0425       weak
         tot_cur_bal     numeric 0.0399       weak
      home_ownership categorical 0.0327       weak
     num_actv_rev_tl     numeric 0.0301       weak
          annual_inc     numeric 0.0296       weak
      inq_last_6mths     numeric 0.0257       weak
mo_sin_old_rev_tl_op     numeric 0.0241       weak
          revol_util     numeric 0.0216       weak
             purpose categorical 0.0181 not useful
             pub_rec     numeri

**What the output shows:**
```
feature        type     IV       band
               grade categorical 0.4726     strong
            int_rate     numeric 0.4627     strong
                term categorical 0.1809     medium
      fico_range_low     numeric 0.1156     medium
                 dti     numeric 0.0747       weak
acc_open_past_24mths     numeric 0.0668       weak
      bc_open_to_buy     numeric 0.0556       weak
 verification_status categorical 0.0548       weak
         avg_cur_bal     numeric 0.0508       weak
            mort_acc     numeric 0.0425       weak
         tot_cur_bal     numeric 0.0399       weak
      home_ownership categorical 0.0327       weak
     num_actv_rev_tl     numeric 0.0301       weak
          annual_inc     numeric 0.0296       weak
      inq_last_6mths     numeric 0.0257       weak
mo_sin_old_rev_tl_op     numeric 0.0241       weak
          revol_util     numeric 0.0216       weak
             purpose categorical 0.0181 not useful
             pub_rec     numeric 0.0068 not useful
            open_acc     numeric 0.0051 not useful
         delinq_2yrs     numeric 0.0049 not useful
           revol_bal     numeric 0.0042 not useful
           total_acc     numeric 0.0017 not useful
```
grade leads at IV=0.4726
(strong band). 6 feature(s)
fall in the "not useful" band -- worth reconsidering whether they're worth
keeping in the `03_processed/` output at all, versus the
4 features in the
medium-or-stronger bands that carry most of the real predictive signal.

**Next:** IV alone doesn't confirm statistical validity. Checking which of
these findings were formally significance-tested (notebook 08) vs. only
descriptively established.

## Cell 3 -- what's statistically confirmed vs. still descriptive

**What / why:** notebook 08 ran formal hypothesis tests on a handful of the
strongest relationships (grade ordering, purpose vs. is_bad, income
distribution, the modeling window itself). Most of the *other* features in
the inventory above were only ever examined descriptively (IV, correlation),
never through a formal significance test. That's not necessarily a problem
given the large sample size, but it's worth being explicit about which
findings in this repo carry a formal p-value and which don't.

**How:** a static reference table, since this reflects what notebook 08
actually tested, not something to recompute.

**Expect:** most core findings are backed by at least one formal test; the
newer notebook-05/09/10/11/13 findings are quasi-causal or descriptive by
design, not hypothesis-tested in the classical sense.

In [3]:
validation_status = pd.DataFrame([
    {"finding": "grade bad-rate ordering", "notebook": "08", "status": "formally tested (pairwise z-tests, all significant)"},
    {"finding": "purpose vs is_bad association", "notebook": "08", "status": "formally tested (chi-square, Bonferroni-robust)"},
    {"finding": "income gap, good vs bad loans", "notebook": "08", "status": "formally tested (Mann-Whitney U)"},
    {"finding": "2013-2017 window appropriateness", "notebook": "08", "status": "formally tested (sensitivity comparison)"},
    {"finding": "baseline model AUC", "notebook": "08", "status": "formally quantified (bootstrap 95% CI)"},
    {"finding": "cluster / anomaly risk differences", "notebook": "05", "status": "descriptive only -- no significance test run"},
    {"finding": "transform / VIF diagnostics", "notebook": "09", "status": "diagnostic, not a hypothesis test"},
    {"finding": "population drift (PSI)", "notebook": "10", "status": "conventional-threshold based, not a p-value test"},
    {"finding": "dti / purpose quasi-causal checks", "notebook": "13", "status": "stratification-based, not a randomized/identified causal estimate"},
])
print(validation_status.to_string(index=False))
validation_status.to_csv(os.path.join(ASSETS_TABLES, "eda14_validation_status.csv"), index=False)


                           finding notebook                                                            status
           grade bad-rate ordering       08               formally tested (pairwise z-tests, all significant)
     purpose vs is_bad association       08                   formally tested (chi-square, Bonferroni-robust)
     income gap, good vs bad loans       08                                  formally tested (Mann-Whitney U)
  2013-2017 window appropriateness       08                          formally tested (sensitivity comparison)
                baseline model AUC       08                            formally quantified (bootstrap 95% CI)
cluster / anomaly risk differences       05                      descriptive only -- no significance test run
       transform / VIF diagnostics       09                                 diagnostic, not a hypothesis test
            population drift (PSI)       10                  conventional-threshold based, not a p-value test
 dti / pur

**What the output shows:**
```
finding notebook                                                            status
           grade bad-rate ordering       08               formally tested (pairwise z-tests, all significant)
     purpose vs is_bad association       08                   formally tested (chi-square, Bonferroni-robust)
     income gap, good vs bad loans       08                                  formally tested (Mann-Whitney U)
  2013-2017 window appropriateness       08                          formally tested (sensitivity comparison)
                baseline model AUC       08                            formally quantified (bootstrap 95% CI)
cluster / anomaly risk differences       05                      descriptive only -- no significance test run
       transform / VIF diagnostics       09                                 diagnostic, not a hypothesis test
            population drift (PSI)       10                  conventional-threshold based, not a p-value test
 dti / purpose quasi-causal checks       13 stratification-based, not a randomized/identified causal estimate
```
The core predictive findings (grade, purpose, income, the window choice, and
the baseline AUC) all carry formal statistical backing. The newer,
methodologically different notebooks (clustering, drift, causal reasoning)
are valuable but use different validation standards -- worth remembering
when citing them: "PSI shows drift" and "z-test confirms significance" are
different strengths of claim, and shouldn't be presented with equal
confidence in a write-up.

**Next:** gathering every caveat and limitation raised across the 13
notebooks into one place, since they're currently scattered one per
notebook.

## Cell 4 -- known caveats and limitations

**What / why:** a reader who only skims this governance notebook should still
walk away knowing every meaningful limitation raised across the suite --
collecting them here means no caveat only lives buried in one notebook's
cell 6.

**How:** a static reference table, curated from the actual findings of
notebooks 01-13.

**Expect:** a compact but complete limitations list.

In [4]:
caveats = pd.DataFrame([
    {"limitation": "matured-only population", "detail": "912,602 still-open loans excluded (notebook 07) -- outcome genuinely unknown, not missing data"},
    {"limitation": "2013-2017 window", "detail": "chosen for reliability/censoring reasons (notebooks 06, 08); windowed vs excluded-years drift is low (notebook 10), so this isn't a representativeness concern"},
    {"limitation": "within-window drift", "detail": "int_rate PSI=0.140 (moderate) across 2013-2017 -- year deserves consideration as a model feature or validation-split dimension (notebook 10)"},
    {"limitation": "grade/int_rate are near-definitional", "detail": "reflect Lending Club's own risk model, not independent causal drivers (notebook 13)"},
    {"limitation": "small-grade precision", "detail": "grade G's bad-rate 95% CI is ~10x wider than grade C's, though still narrow in absolute terms (notebook 11)"},
    {"limitation": "geographic tail", "detail": "smallest states have single-digit loan counts -- exclude from any per-state analysis (notebook 11)"},
    {"limitation": "emp_title unusable as-is", "detail": "317,489 distinct values, 28.3% distinct-per-row -- needs occupation-taxonomy mapping before use (notebook 12)"},
    {"limitation": "multicollinearity", "detail": "tot_cur_bal and avg_cur_bal show elevated VIF -- consider collapsing before a linear model (notebook 09)"},
    {"limitation": "log transform not universal", "detail": "improved target correlation for 9 of 16 transformed fields, not all 16 (notebook 09)"},
    {"limitation": "recoveries not captured by is_bad", "detail": "charged-off loans average partial recovery -- is_bad is a pure PD target, not a loss-given-default one (notebook 07)"},
])
print(caveats.to_string(index=False))
caveats.to_csv(os.path.join(ASSETS_TABLES, "eda14_caveats.csv"), index=False)


                          limitation                                                                                                                                                         detail
             matured-only population                                                                 912,602 still-open loans excluded (notebook 07) -- outcome genuinely unknown, not missing data
                    2013-2017 window chosen for reliability/censoring reasons (notebooks 06, 08); windowed vs excluded-years drift is low (notebook 10), so this isn't a representativeness concern
                 within-window drift                   int_rate PSI=0.140 (moderate) across 2013-2017 -- year deserves consideration as a model feature or validation-split dimension (notebook 10)
grade/int_rate are near-definitional                                                                            reflect Lending Club's own risk model, not independent causal drivers (notebook 13)
               small

**What the output shows:**
```
limitation                                                                                                                                                         detail
             matured-only population                                                                 912,602 still-open loans excluded (notebook 07) -- outcome genuinely unknown, not missing data
                    2013-2017 window chosen for reliability/censoring reasons (notebooks 06, 08); windowed vs excluded-years drift is low (notebook 10), so this isn't a representativeness concern
                 within-window drift                   int_rate PSI=0.140 (moderate) across 2013-2017 -- year deserves consideration as a model feature or validation-split dimension (notebook 10)
grade/int_rate are near-definitional                                                                            reflect Lending Club's own risk model, not independent causal drivers (notebook 13)
               small-grade precision                                                    grade G's bad-rate 95% CI is ~10x wider than grade C's, though still narrow in absolute terms (notebook 11)
                     geographic tail                                                             smallest states have single-digit loan counts -- exclude from any per-state analysis (notebook 11)
            emp_title unusable as-is                                                  317,489 distinct values, 28.3% distinct-per-row -- needs occupation-taxonomy mapping before use (notebook 12)
                   multicollinearity                                                       tot_cur_bal and avg_cur_bal show elevated VIF -- consider collapsing before a linear model (notebook 09)
         log transform not universal                                                                           improved target correlation for 9 of 16 transformed fields, not all 16 (notebook 09)
   recoveries not captured by is_bad                                           charged-off loans average partial recovery -- is_bad is a pure PD target, not a loss-given-default one (notebook 07)
```
10 limitations, none of them disqualifying -- every one is a
"use with this caveat" item, not a "this data can't be trusted" item. That's
itself a useful conclusion: the dataset is solid enough to build on, provided
Phase 1 respects these specific caveats rather than treating the data as
caveat-free.

**Next:** turning caveats and the feature inventory into concrete action
items for the `03_data_cleaning` rebuild (task 11).

## Cell 5 -- action items for the data-cleaning rebuild

**What / why:** this is the direct hand-off to task 11 (rebuilding
`03_data_cleaning/01_cleaning_and_feature_prep.ipynb`) -- translating
everything found in notebooks 05-13 into concrete, checkable actions, so the
rebuild is grounded in evidence rather than repeating the original guesses
made before this deeper EDA existed.

**How:** a static action list, derived directly from the diagnostic results
in notebook 09 and the inventory in cell 2 above.

**Expect:** a short, concrete checklist.

In [5]:
cleaning_actions = pd.DataFrame([
    {"action": "keep median imputation approach", "reason": "notebooks 02/09 found no evidence to change the original imputation logic"},
    {"action": "reconsider log transform for 7 fields", "reason": "acc_open_past_24mths, avg_cur_bal, bc_open_to_buy, num_actv_rev_tl, open_acc, revol_bal, tot_cur_bal did not improve target correlation (notebook 09)"},
    {"action": "consider dropping or collapsing tot_cur_bal / avg_cur_bal", "reason": "elevated VIF, likely redundant (notebook 09)"},
    {"action": "add a year or vintage feature", "reason": "int_rate PSI=0.140 within the window -- year carries real signal (notebook 10)"},
    {"action": "flag / exclude smallest states from state-level features", "reason": "single-digit loan counts in the smallest states (notebook 11)"},
    {"action": "do not include raw emp_title as a feature", "reason": "unusable without occupation-taxonomy normalization (notebook 12)"},
    {"action": "keep grade and int_rate, but document as near-definitional", "reason": "not independent causal drivers -- fine as predictors, misleading if described as causal (notebook 13)"},
])
print(cleaning_actions.to_string(index=False))
cleaning_actions.to_csv(os.path.join(ASSETS_TABLES, "eda14_cleaning_actions.csv"), index=False)


                                                    action                                                                                                                                                reason
                           keep median imputation approach                                                                             notebooks 02/09 found no evidence to change the original imputation logic
                     reconsider log transform for 7 fields acc_open_past_24mths, avg_cur_bal, bc_open_to_buy, num_actv_rev_tl, open_acc, revol_bal, tot_cur_bal did not improve target correlation (notebook 09)
 consider dropping or collapsing tot_cur_bal / avg_cur_bal                                                                                                          elevated VIF, likely redundant (notebook 09)
                             add a year or vintage feature                                                                        int_rate PSI=0.140 within the wind

**What the output shows:**
```
action                                                                                                                                                reason
                           keep median imputation approach                                                                             notebooks 02/09 found no evidence to change the original imputation logic
                     reconsider log transform for 7 fields acc_open_past_24mths, avg_cur_bal, bc_open_to_buy, num_actv_rev_tl, open_acc, revol_bal, tot_cur_bal did not improve target correlation (notebook 09)
 consider dropping or collapsing tot_cur_bal / avg_cur_bal                                                                                                          elevated VIF, likely redundant (notebook 09)
                             add a year or vintage feature                                                                        int_rate PSI=0.140 within the window -- year carries real signal (notebook 10)
  flag / exclude smallest states from state-level features                                                                                         single-digit loan counts in the smallest states (notebook 11)
                 do not include raw emp_title as a feature                                                                                      unusable without occupation-taxonomy normalization (notebook 12)
keep grade and int_rate, but document as near-definitional                                                 not independent causal drivers -- fine as predictors, misleading if described as causal (notebook 13)
```
7 concrete, evidence-backed changes to apply when
task 11 rebuilds the cleaning notebook -- each one traceable to a specific
EDA notebook's finding, not a guess.

**Next:** the last piece -- open questions and action items that belong to
Phase 1 rather than to Phase 0's cleaning step.

## Cell 6 -- action items and open questions for Phase 1

**What / why:** not every finding in this suite is actionable within Phase
0's data-cleaning scope -- some are genuinely modeling decisions (validation
strategy, whether to engineer cluster membership as a feature) that belong
downstream. Listing them here means Phase 1 starts with a checklist instead
of having to re-derive it from 14 notebooks.

**How:** a static list, closing out this notebook and the EDA suite.

**Expect:** the final cell in the Lending Club EDA suite.

In [6]:
phase1_items = pd.DataFrame([
    {"item": "validation strategy", "detail": "consider year-stratified or time-based validation, not pure random split, given within-window drift (notebook 10)"},
    {"item": "cluster membership / anomaly score as features", "detail": "MiniBatchKMeans clusters and Isolation Forest anomaly scores both showed real bad-rate spread (notebook 05) -- worth testing as engineered features"},
    {"item": "sub_grade", "detail": "not in the current feature set; notebook 13 suggests int_rate's within-grade signal likely comes from sub_grade -- consider adding it directly"},
    {"item": "loss-given-default modeling", "detail": "is_bad is PD-only; recovery amounts among charged-off loans are substantial and variable (notebook 07) -- a separate LGD model may be worth scoping"},
    {"item": "occupation feature engineering", "detail": "emp_title needs a taxonomy-mapping effort before it's usable (notebook 12) -- worth scoping if time allows"},
    {"item": "grade/int_rate framing in any write-up", "detail": "describe as definitional risk-tier features, not causal drivers (notebook 13)"},
])
print(phase1_items.to_string(index=False))
phase1_items.to_csv(os.path.join(ASSETS_TABLES, "eda14_phase1_items.csv"), index=False)
print()
print(f"EDA suite complete: 14 of 14 categories addressed (6 built to full analytical depth,")
print("8 scaffolded or built proportional to available content) for Lending Club.")


                                          item                                                                                                                                              detail
                           validation strategy                                   consider year-stratified or time-based validation, not pure random split, given within-window drift (notebook 10)
cluster membership / anomaly score as features MiniBatchKMeans clusters and Isolation Forest anomaly scores both showed real bad-rate spread (notebook 05) -- worth testing as engineered features
                                     sub_grade      not in the current feature set; notebook 13 suggests int_rate's within-grade signal likely comes from sub_grade -- consider adding it directly
                   loss-given-default modeling is_bad is PD-only; recovery amounts among charged-off loans are substantial and variable (notebook 07) -- a separate LGD model may be worth scoping
                occupatio

**What the output shows:**
```
item                                                                                                                                              detail
                           validation strategy                                   consider year-stratified or time-based validation, not pure random split, given within-window drift (notebook 10)
cluster membership / anomaly score as features MiniBatchKMeans clusters and Isolation Forest anomaly scores both showed real bad-rate spread (notebook 05) -- worth testing as engineered features
                                     sub_grade      not in the current feature set; notebook 13 suggests int_rate's within-grade signal likely comes from sub_grade -- consider adding it directly
                   loss-given-default modeling is_bad is PD-only; recovery amounts among charged-off loans are substantial and variable (notebook 07) -- a separate LGD model may be worth scoping
                occupation feature engineering                                          emp_title needs a taxonomy-mapping effort before it's usable (notebook 12) -- worth scoping if time allows
        grade/int_rate framing in any write-up                                                                       describe as definitional risk-tier features, not causal drivers (notebook 13)

EDA suite complete: 14 of 14 categories addressed (6 built to full analytical depth,
8 scaffolded or built proportional to available content) for Lending Club.
```
This closes out the Lending Club EDA suite. Per this project's build order,
the next step is NOT another dataset -- it's rebuilding
`03_data_cleaning/01_cleaning_and_feature_prep.ipynb` using the cleaning
action items from cell 5, validating the new `data/03_processed/` output,
and only then closing out Lending Club's Phase 0 entirely before any other
dataset's work begins.

**Next:** task 11 -- rebuild the cleaning notebook.

## Cell 7 -- categorical encoding strategy (gap-closure addendum)

**What / why:** an audit of this EDA suite found one documentation gap:
`data/03_processed/lendingclub_model_ready.parquet` keeps every categorical
field as its original string value -- deliberately, since encoding choice
(one-hot vs. ordinal vs. target/frequency encoding) is a modeling decision
that depends on which model family Phase 1 ends up using, and baking one
choice into Phase 0 would force every Phase 1 model to live with it. That
was always the intent, but it was never written down anywhere -- this cell
fixes that by recording the actual decision, per field, with the cardinality
that decision should be checked against.

**How:** pull real distinct-value counts (on `windowed`, the same population
the cleaning notebook uses) for every retained categorical field plus the
one engineered categorical (`addr_state_grouped` is a cleaning-notebook
output, not a `windowed` column, so its cardinality is taken from that
notebook's validated result instead of queried here), and pair each with a
recommended Phase-1 encoding approach and the reasoning behind it.

**Expect:** low-cardinality fields (`term`, `home_ownership`,
`verification_status`) are cheap to one-hot regardless of model choice;
`grade` has a real ordinal structure worth preserving rather than discarding
into one-hot; `purpose` and `addr_state` are the two fields where cardinality
actually matters for the encoding choice.

In [7]:
CATEGORICAL_FIELDS = ["term", "grade", "emp_length", "home_ownership",
                      "verification_status", "purpose", "addr_state"]

card_rows = []
for c in CATEGORICAL_FIELDS:
    n_distinct = con.sql(f'SELECT count(DISTINCT "{c}") FROM windowed').fetchone()[0]
    card_rows.append({"field": c, "n_distinct_values": n_distinct})
# addr_state_grouped is produced in the cleaning notebook (states with <1000
# windowed loans collapsed to 'OTHER'), not present in windowed itself --
# its cardinality is the validated result from that notebook: only one raw
# addr_state value (IA, 2 loans total) qualified for grouping, so the
# distinct count is unchanged (IA is replaced by 'OTHER', 1-for-1).
card_rows.append({"field": "addr_state_grouped (engineered in 03_data_cleaning)", "n_distinct_values": 51})
card_df = pd.DataFrame(card_rows)

ENCODING_DECISION = {
    "term": ("binary indicator (is_60_month)", "only 2 values (36/60 months) -- a single 0/1 flag carries the same information as one-hot with no redundant column"),
    "grade": ("ordinal (A=1 ... G=7)", "grade is Lending Club's own ordered risk tier (notebook 13) -- collapsing that order into unordered one-hot columns throws away real structure a tree or linear model can otherwise use directly"),
    "emp_length": ("ordinal (0-10, missing as its own level)", "years-of-employment has a natural order, and notebook 02 found the missing group itself carries signal (higher bad rate) -- worth keeping as an explicit level, not imputing it away"),
    "home_ownership": ("one-hot", "small, unordered category set (RENT/OWN/MORTGAGE/OTHER-family) -- no natural order, low enough cardinality that one-hot adds few columns"),
    "verification_status": ("one-hot", "3 unordered levels -- same reasoning as home_ownership"),
    "purpose": ("one-hot, or frequency/target encoding if the model family penalizes wide one-hot", "moderate cardinality (~14 values), chi-square-confirmed association with is_bad (notebook 08) but a largely confounded-by-grade effect (notebook 13) -- one-hot is safe for tree models; a linear/regularized model may prefer a lower-dimensional encoding"),
    "addr_state": ("drop in favor of addr_state_grouped, then one-hot or target encoding", "50+ raw levels is too wide for one-hot on a linear model and too sparse in the tail states (notebook 11) for target encoding to be stable without the small-state grouping already applied in 03_data_cleaning"),
    "addr_state_grouped (engineered in 03_data_cleaning)": ("one-hot for tree models; target/frequency encoding worth testing for linear models", "same field as addr_state but with the single sub-1000-loan state (IA) collapsed to 'OTHER' -- removes the one cell too sparse for stable target encoding"),
}
card_df["recommended_encoding"] = card_df["field"].map(lambda f: ENCODING_DECISION[f][0])
card_df["reasoning"] = card_df["field"].map(lambda f: ENCODING_DECISION[f][1])

print(card_df[["field", "n_distinct_values", "recommended_encoding"]].to_string(index=False))
print()
for _, r in card_df.iterrows():
    print(f"- {r['field']}: {r['reasoning']}")
card_df.to_csv(os.path.join(ASSETS_TABLES, "eda14_card_df.csv"), index=False)


                                              field  n_distinct_values                                                               recommended_encoding
                                               term                  2                                                     binary indicator (is_60_month)
                                              grade                  7                                                              ordinal (A=1 ... G=7)
                                         emp_length                 11                                           ordinal (0-10, missing as its own level)
                                     home_ownership                  5                                                                            one-hot
                                verification_status                  3                                                                            one-hot
                                            purpose                 14   one

**What the output shows:**
```
field  n_distinct_values                                                               recommended_encoding
                                               term                  2                                                     binary indicator (is_60_month)
                                              grade                  7                                                              ordinal (A=1 ... G=7)
                                         emp_length                 11                                           ordinal (0-10, missing as its own level)
                                     home_ownership                  5                                                                            one-hot
                                verification_status                  3                                                                            one-hot
                                            purpose                 14   one-hot, or frequency/target encoding if the model family penalizes wide one-hot
                                         addr_state                 51               drop in favor of addr_state_grouped, then one-hot or target encoding
addr_state_grouped (engineered in 03_data_cleaning)                 51 one-hot for tree models; target/frequency encoding worth testing for linear models

- term: only 2 values (36/60 months) -- a single 0/1 flag carries the same information as one-hot with no redundant column
- grade: grade is Lending Club's own ordered risk tier (notebook 13) -- collapsing that order into unordered one-hot columns throws away real structure a tree or linear model can otherwise use directly
- emp_length: years-of-employment has a natural order, and notebook 02 found the missing group itself carries signal (higher bad rate) -- worth keeping as an explicit level, not imputing it away
- home_ownership: small, unordered category set (RENT/OWN/MORTGAGE/OTHER-family) -- no natural order, low enough cardinality that one-hot adds few columns
- verification_status: 3 unordered levels -- same reasoning as home_ownership
- purpose: moderate cardinality (~14 values), chi-square-confirmed association with is_bad (notebook 08) but a largely confounded-by-grade effect (notebook 13) -- one-hot is safe for tree models; a linear/regularized model may prefer a lower-dimensional encoding
- addr_state: 50+ raw levels is too wide for one-hot on a linear model and too sparse in the tail states (notebook 11) for target encoding to be stable without the small-state grouping already applied in 03_data_cleaning
- addr_state_grouped (engineered in 03_data_cleaning): same field as addr_state but with the single sub-1000-loan state (IA) collapsed to 'OTHER' -- removes the one cell too sparse for stable target encoding
```
This is the categorical-encoding decision that was implicit but undocumented:
`03_data_cleaning` deliberately leaves every categorical field as its raw
string value in `lendingclub_model_ready.parquet`, and encoding is deferred
to Phase 1 by design -- not an oversight. The table above is the actual
per-field recommendation Phase 1 should start from, grounded in each field's
real cardinality and in what the rest of this EDA suite already established
about it (`grade`'s ordinal structure from notebook 13, `emp_length`'s
informative-missingness from notebook 02, `addr_state`'s sparse-tail problem
from notebook 11).

**Next:** this was the fourth and final gap identified in the depth/
sufficiency review -- duplicates, leakage re-validation, and outlier
treatment were closed in `02_data_quality_integrity.ipynb`'s cells 6-8; this
cell closes the encoding-strategy documentation gap. All four gaps are now
closed.